In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/.git-blame-ignore-revs
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/.pre-commit-config.yaml
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/AUTHORS.rst
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/NOTICE
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/LICENSE
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/.gitignore
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/requirements-dev.txt
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/Makefile
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/HISTORY.md
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/pyproject.toml
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/README.md
/kaggle/input/datasets/samsonoluwadare/request-repo/requests-main/MANIFEST.in
/kaggle/input/datasets/samsonoluwadare/reques

In [ ]:
# CodeGraph-Agent: Graph-based Bug Localization for Gemma 4

This notebook demonstrates **CodeGraph-Priority**, a method for ranking functions in a Python repository based on:
- Text similarity between a bug description and function documentation.
- Graph-based importance derived from the call graph (PageRank).

## What this notebook does

- Builds a typed code graph (files + functions, CALLS + IMPORTS edges) from a Python repository.
- Ranks functions for a given natural-language bug query.
- Demonstrates the approach on two synthetic repositories:
  - A minimal 4-file example.
  - A larger 8-file ML-style project.

## How to reuse this code

1. Replace the demo repository path with your own Python repo directory.
2. Call:
   - `build_repo_code_graph(repo_dir)` to construct the graph.
   - `extract_function_texts(repo_dir)` to get function metadata.
   - `rank_functions_for_bug(G, func_info, bug_description, top_k=5)` to get top-k functions.
3. Use the top-k functions to construct prompts for Gemma 4 or other LLMs.

The code is intended as a reusable toolkit for research on graph-aware agentic software engineering.

In [2]:
# Reinstall dependencies (quietly)
!pip install networkx scikit-learn sentence-transformers -q

# Try importing them
import networkx as nx
import numpy as np
from sentence_transformers import SentenceTransformer

print("networkx version:", nx.__version__)
print("numpy version:", np.__version__)
print("sentence-transformers imported successfully")

networkx version: 3.6.1
numpy version: 2.1.3
sentence-transformers imported successfully


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(["Hello, world!"], convert_to_numpy=True, normalize_embeddings=True)
print("Embedding shape:", emb.shape)
print("Sample embedding (first 5 dims):", emb[0, :5])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (1, 384)
Sample embedding (first 5 dims): [-0.03817716  0.03291108 -0.00545938  0.01436989 -0.04029098]


In [4]:
import tempfile
import os

def create_tiny_demo_repo(base_dir: str):
    os.makedirs(os.path.join(base_dir, "pkg"), exist_ok=True)

    with open(os.path.join(base_dir, "pkg", "utils.py"), "w") as f:
        f.write("""
def helper(x):
    \"\"\"Add one to x.\"\"\"
    return x + 1

def process(x):
    \"\"\"Process x using helper.\"\"\"
    return helper(x) * 2
""")

    with open(os.path.join(base_dir, "pkg", "main.py"), "w") as f:
        f.write("""
from pkg.utils import helper, process

def run():
    \"\"\"Run the main computation.\"\"\"
    x = 10
    y = process(x)
    return y

if __name__ == "__main__":
    run()
""")

tmp_demo_repo = tempfile.mkdtemp()
create_tiny_demo_repo(tmp_demo_repo)
print("Demo repo created at:", tmp_demo_repo)

Demo repo created at: /tmp/tmpb4qok1f3


In [5]:
# Re-import in case you restarted the session
import ast
import os
from typing import Dict, List, Set, Tuple, Optional, Any
import networkx as nx
import numpy as np
from sentence_transformers import SentenceTransformer

# Paste the full graph builder code again (needed if this is a fresh run)

class CodeGraphBuilder(ast.NodeVisitor):
    def __init__(self, module_name: str, file_node_id: str):
        self.module_name = module_name
        self.file_node_id = file_node_id
        self.current_function: Optional[str] = None
        self.calls: List[Tuple[str, str]] = []
        self.imported_files: Set[str] = set()

    def visit_FunctionDef(self, node: ast.FunctionDef):
        old_function = self.current_function
        self.current_function = f"func:{self.module_name}.{node.name}"
        self.generic_visit(node)
        self.current_function = old_function

    visit_AsyncFunctionDef = visit_FunctionDef

    def visit_Call(self, node: ast.Call):
        if self.current_function is None:
            self.generic_visit(node)
            return

        callee = self._resolve_call(node.func)
        if callee is not None:
            self.calls.append((self.current_function, callee))

        self.generic_visit(node)

    def visit_Import(self, node: ast.Import):
        for alias in node.names:
            mod_name = alias.name.split(".")[0]
            file_node = self._module_to_file_node(mod_name)
            if file_node is not None:
                self.imported_files.add(file_node)
        self.generic_visit(node)

    def visit_ImportFrom(self, node: ast.ImportFrom):
        if node.module is None:
            self.generic_visit(node)
            return
        mod_name = node.module.split(".")[0]
        file_node = self._module_to_file_node(mod_name)
        if file_node is not None:
            self.imported_files.add(file_node)
        self.generic_visit(node)

    def _resolve_call(self, func_node: ast.AST) -> Optional[str]:
        if isinstance(func_node, ast.Name):
            return f"func:{self.module_name}.{func_node.id}"
        elif isinstance(func_node, ast.Attribute):
            parts = []
            node_ = func_node
            while isinstance(node_, ast.Attribute):
                parts.append(node_.attr)
                node_ = node_.value
            if isinstance(node_, ast.Name):
                parts.append(node_.id)
                parts.reverse()
                return f"func:{self.module_name}.{'.'.join(parts)}"
        return None

    def _module_to_file_node(self, mod_name: str) -> Optional[str]:
        return f"file:{mod_name}"


def build_code_graph_for_file(file_path: str, module_name: str) -> Tuple[str, List[Tuple[str, str]], Set[str]]:
    with open(file_path, "r", encoding="utf-8") as f:
        source = f.read()
    tree = ast.parse(source, filename=file_path)
    file_node_id = f"file:{module_name}"
    visitor = CodeGraphBuilder(module_name, file_node_id)
    visitor.visit(tree)
    return file_node_id, visitor.calls, visitor.imported_files


def build_repo_code_graph(repo_dir: str) -> nx.DiGraph:
    G = nx.DiGraph()
    py_files = []

    for root, _, files in os.walk(repo_dir):
        for fname in files:
            if fname.endswith(".py"):
                py_files.append(os.path.join(root, fname))

    repo_dir_abs = os.path.abspath(repo_dir)

    file_info = []
    for full_path in py_files:
        rel_path = os.path.relpath(full_path, repo_dir_abs)
        module_name = rel_path.replace(os.sep, ".").rsplit(".py", 1)[0]
        file_info.append((full_path, module_name))

    module_prefix_to_file = {}
    for full_path, module_name in file_info:
        parts = module_name.split(".")
        for i in range(1, len(parts)+1):
            prefix = ".".join(parts[:i])
            module_prefix_to_file[prefix] = f"file:{module_name}"

    for full_path, module_name in file_info:
        file_node_id, calls, imported_modules = build_code_graph_for_file(full_path, module_name)

        G.add_node(file_node_id, type="file", module=module_name, path=full_path)

        for caller, callee in calls:
            if not G.has_node(caller):
                parts = caller[len("func:"):].rsplit(".", 1)
                if len(parts) == 2:
                    mod, func_name = parts
                    G.add_node(caller, type="function", module=mod, name=func_name)
            if not G.has_node(callee):
                parts = callee[len("func:"):].rsplit(".", 1)
                if len(parts) == 2:
                    mod, func_name = parts
                    G.add_node(callee, type="function", module=mod, name=func_name)
            G.add_edge(caller, callee, type="CALLS")

        for mod_prefix in imported_modules:
            target_file = module_prefix_to_file.get(mod_prefix)
            if target_file is None:
                target_file = f"file:{mod_prefix}"
            G.add_edge(file_node_id, target_file, type="IMPORTS")

    return G

In [6]:
from typing import Any

def extract_function_texts(repo_dir: str) -> Dict[str, Dict[str, Any]]:
    func_info = {}
    for root, _, files in os.walk(repo_dir):
        for fname in files:
            if not fname.endswith(".py"):
                continue
            full_path = os.path.join(root, fname)
            rel_path = os.path.relpath(full_path, os.path.abspath(repo_dir))
            module_name = rel_path.replace(os.sep, ".").rsplit(".py", 1)[0]

            with open(full_path, "r", encoding="utf-8") as f:
                source = f.read()
            tree = ast.parse(source, filename=full_path)

            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    func_id = f"func:{module_name}.{node.name}"
                    sig = f"def {node.name}(...):"
                    doc = ast.get_docstring(node)
                    func_info[func_id] = {
                        "module": module_name,
                        "name": node.name,
                        "signature": sig,
                        "docstring": doc,
                        "file": full_path,
                    }
    return func_info

In [7]:
def rank_functions_for_bug(
    G: nx.DiGraph,
    func_info: Dict[str, Dict[str, Any]],
    bug_description: str,
    top_k: int = 5,
    alpha: float = 0.7,
) -> List[str]:
    func_ids = [fid for fid in func_info if fid in G.nodes()]
    if len(func_ids) == 0:
        return []

    texts = []
    for fid in func_ids:
        info = func_info[fid]
        text_parts = [info["signature"]]
        if info["docstring"]:
            text_parts.append(info["docstring"])
        texts.append("\n".join(text_parts))

    model = SentenceTransformer("all-MiniLM-L6-v2")
    bug_emb = model.encode([bug_description], convert_to_numpy=True, normalize_embeddings=True)
    func_embs = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

    sims = (bug_emb @ func_embs.T).flatten()

    call_subgraph = G.subgraph([n for n, d in G.nodes(data=True) if d.get("type") == "function"]).copy()
    edges_to_keep = [(u, v) for u, v, d in call_subgraph.edges(data=True) if d.get("type") == "CALLS"]
    call_subgraph = call_subgraph.edge_subgraph(edges_to_keep).copy()

    if len(call_subgraph.nodes()) == 0:
        pr = {fid: 0.0 for fid in func_ids}
    else:
        pr = nx.pagerank(call_subgraph, max_iter=100)

    pr_scores = np.array([pr.get(fid, 0.0) for fid in func_ids])
    if pr_scores.max() > 0:
        pr_scores = pr_scores / pr_scores.max()

    final_scores = alpha * sims + (1 - alpha) * pr_scores
    top_indices = np.argsort(final_scores)[::-1][:top_k]
    top_funcs = [func_ids[i] for i in top_indices]
    return top_funcs

In [8]:
# tmp_demo_repo should still exist from your earlier cell
G_demo = build_repo_code_graph(tmp_demo_repo)
func_info_demo = extract_function_texts(tmp_demo_repo)

print("Function nodes:")
for n, d in G_demo.nodes(data=True):
    if d.get("type") == "function":
        print(n, d)

print("\nCALLS edges:")
for u, v, d in G_demo.edges(data=True):
    if d.get("type") == "CALLS":
        print(u, "->", v)

print("\nIMPORTS edges:")
for u, v, d in G_demo.edges(data=True):
    if d.get("type") == "IMPORTS":
        print(u, "->", v)

Function nodes:
func:pkg.main.run {'type': 'function', 'module': 'pkg.main', 'name': 'run'}
func:pkg.main.process {'type': 'function', 'module': 'pkg.main', 'name': 'process'}
func:pkg.utils.process {'type': 'function', 'module': 'pkg.utils', 'name': 'process'}
func:pkg.utils.helper {'type': 'function', 'module': 'pkg.utils', 'name': 'helper'}

CALLS edges:
func:pkg.main.run -> func:pkg.main.process
func:pkg.utils.process -> func:pkg.utils.helper

IMPORTS edges:
file:pkg.main -> file:file:pkg


In [9]:
bug_query = "Fix the off-by-one error in the processing loop."

top_funcs = rank_functions_for_bug(G_demo, func_info_demo, bug_query, top_k=3)

print("Top functions for bug:")
for fid in top_funcs:
    info = func_info_demo[fid]
    print("ID:", fid)
    print("  File:", info["file"])
    print("  Signature:", info["signature"])
    print("  Docstring:", info["docstring"])
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top functions for bug:
ID: func:pkg.utils.process
  File: /tmp/tmpb4qok1f3/pkg/utils.py
  Signature: def process(...):
  Docstring: Process x using helper.

ID: func:pkg.utils.helper
  File: /tmp/tmpb4qok1f3/pkg/utils.py
  Signature: def helper(...):
  Docstring: Add one to x.

ID: func:pkg.main.run
  File: /tmp/tmpb4qok1f3/pkg/main.py
  Signature: def run(...):
  Docstring: Run the main computation.



In [10]:
import os

kaggle_input_root = "/kaggle/input"

print("Contents of /kaggle/input:")
if os.path.exists(kaggle_input_root):
    for name in os.listdir(kaggle_input_root):
        full_path = os.path.join(kaggle_input_root, name)
        print(name)
        if os.path.isdir(full_path):
            for sub in os.listdir(full_path):
                print("  ", sub)
else:
    print("/kaggle/input does not exist")

Contents of /kaggle/input:
datasets
   samsonoluwadare
competitions
   gemma-4-developer-agent-paper


In [11]:
import os
import tempfile

def create_realistic_demo_repo(base_dir: str):
    # Structure:
    # repo/
    #   data_utils.py
    #   model.py
    #   train.py
    #   eval.py

    with open(os.path.join(base_dir, "data_utils.py"), "w") as f:
        f.write("""
import json

def load_data(path: str):
    \"\"\"Load JSON data from a file.\"\"\"
    with open(path, "r") as f:
        return json.load(f)

def preprocess_data(data):
    \"\"\"Preprocess raw data before feeding to model.\"\"\"
    # Simple normalization
    return [x / 255.0 for x in data]

def save_predictions(path, preds):
    \"\"\"Save predictions to a JSON file.\"\"\"
    with open(path, "w") as f:
        json.dump(preds, f)
""")

    with open(os.path.join(base_dir, "model.py"), "w") as f:
        f.write("""
from data_utils import preprocess_data

def build_model(input_dim: int, hidden_dim: int = 64):
    \"\"\"Build a simple MLP model (mock implementation).\"\"\"
    # In a real repo, this would construct layers
    return {"input_dim": input_dim, "hidden_dim": hidden_dim}

def forward(model, x):
    \"\"\"Run a forward pass through the model.\"\"\"
    x = preprocess_data(x)
    # Mock forward: just a linear transform
    h = sum(x) * model["hidden_dim"]
    return h

def train_step(model, x, y, lr=0.01):
    \"\"\"Perform a single training step.\"\"\"
    pred = forward(model, x)
    loss = (pred - y) ** 2
    # Mock gradient update
    model["hidden_dim"] -= lr * loss
    return loss
""")

    with open(os.path.join(base_dir, "train.py"), "w") as f:
        f.write("""
from data_utils import load_data, save_predictions
from model import build_model, train_step

def train_pipeline(data_path: str, model_path: str):
    \"\"\"Main training pipeline.\"\"\"
    data = load_data(data_path)
    model = build_model(input_dim=10, hidden_dim=32)
    losses = []
    for x, y in data:
        loss = train_step(model, x, y)
        losses.append(loss)
    save_predictions(model_path, losses)
    return model, losses
""")

    with open(os.path.join(base_dir, "eval.py"), "w") as f:
        f.write("""
from data_utils import load_data
from model import build_model, forward

def evaluate(model_path: str, data_path: str):
    \"\"\"Evaluate the trained model on a dataset.\"\"\"
    # In a real system, you'd load model weights here
    model = build_model(input_dim=10, hidden_dim=32)
    data = load_data(data_path)
    preds = []
    for x, _ in data:
        pred = forward(model, x)
        preds.append(pred)
    return preds
""")

# Create the repo in a temp directory
realistic_repo_dir = tempfile.mkdtemp()
create_realistic_demo_repo(realistic_repo_dir)

print("Realistic demo repo created at:", realistic_repo_dir)
print("Top-level files:", os.listdir(realistic_repo_dir))

Realistic demo repo created at: /tmp/tmpd8_1ngdp
Top-level files: ['train.py', 'eval.py', 'data_utils.py', 'model.py']


In [12]:
# Build graph
G_realistic = build_repo_code_graph(realistic_repo_dir)

# Extract function texts
func_info_realistic = extract_function_texts(realistic_repo_dir)

print("Number of nodes:", G_realistic.number_of_nodes())
print("Number of edges:", G_realistic.number_of_edges())
print("Number of functions:", len(func_info_realistic))

print("\nFunction nodes:")
for n, d in G_realistic.nodes(data=True):
    if d.get("type") == "function":
        print(n)

print("\nCALLS edges:")
for u, v, d in G_realistic.edges(data=True):
    if d.get("type") == "CALLS":
        print(u, "->", v)

print("\nIMPORTS edges:")
for u, v, d in G_realistic.edges(data=True):
    if d.get("type") == "IMPORTS":
        print(u, "->", v)

Number of nodes: 27
Number of edges: 22
Number of functions: 8

Function nodes:
func:train.train_pipeline
func:train.load_data
func:train.build_model
func:train.train_step
func:train.losses.append
func:train.save_predictions
func:eval.evaluate
func:eval.build_model
func:eval.load_data
func:eval.forward
func:eval.preds.append
func:data_utils.load_data
func:data_utils.open
func:data_utils.json.load
func:data_utils.save_predictions
func:data_utils.json.dump
func:model.forward
func:model.preprocess_data
func:model.sum
func:model.train_step

CALLS edges:
func:train.train_pipeline -> func:train.load_data
func:train.train_pipeline -> func:train.build_model
func:train.train_pipeline -> func:train.train_step
func:train.train_pipeline -> func:train.losses.append
func:train.train_pipeline -> func:train.save_predictions
func:eval.evaluate -> func:eval.build_model
func:eval.evaluate -> func:eval.load_data
func:eval.evaluate -> func:eval.forward
func:eval.evaluate -> func:eval.preds.append
func:data

In [13]:
bug_queries = [
    "Fix a bug where the loss is not saved correctly.",
    "Preprocessing does not normalize the data properly.",
    "Model forward pass gives wrong predictions.",
]

for bug in bug_queries:
    print("Bug query:", bug)
    top_funcs = rank_functions_for_bug(G_realistic, func_info_realistic, bug, top_k=3, alpha=0.7)
    print("Top 3 functions:")
    for fid in top_funcs:
        info = func_info_realistic[fid]
        print("  -", fid)
        print("    File:", info["file"])
        print("    Signature:", info["signature"])
        if info["docstring"]:
            print("    Docstring:", info["docstring"])
    print()

Bug query: Fix a bug where the loss is not saved correctly.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 3 functions:
  - func:model.forward
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def forward(...):
    Docstring: Run a forward pass through the model.
  - func:data_utils.save_predictions
    File: /tmp/tmpd8_1ngdp/data_utils.py
    Signature: def save_predictions(...):
    Docstring: Save predictions to a JSON file.
  - func:model.train_step
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def train_step(...):
    Docstring: Perform a single training step.

Bug query: Preprocessing does not normalize the data properly.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 3 functions:
  - func:model.forward
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def forward(...):
    Docstring: Run a forward pass through the model.
  - func:train.train_pipeline
    File: /tmp/tmpd8_1ngdp/train.py
    Signature: def train_pipeline(...):
    Docstring: Main training pipeline.
  - func:data_utils.load_data
    File: /tmp/tmpd8_1ngdp/data_utils.py
    Signature: def load_data(...):
    Docstring: Load JSON data from a file.

Bug query: Model forward pass gives wrong predictions.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 3 functions:
  - func:model.forward
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def forward(...):
    Docstring: Run a forward pass through the model.
  - func:data_utils.save_predictions
    File: /tmp/tmpd8_1ngdp/data_utils.py
    Signature: def save_predictions(...):
    Docstring: Save predictions to a JSON file.
  - func:eval.evaluate
    File: /tmp/tmpd8_1ngdp/eval.py
    Signature: def evaluate(...):
    Docstring: Evaluate the trained model on a dataset.



In [14]:
bug_queries = [
    "Fix a bug where the loss is not saved correctly.",
    "Preprocessing does not normalize the data properly.",
    "Model forward pass gives wrong predictions.",
]

for bug in bug_queries:
    print("Bug query:", bug)
    top_funcs = rank_functions_for_bug(G_realistic, func_info_realistic, bug, top_k=3, alpha=0.7)
    print("Top 3 functions:")
    for fid in top_funcs:
        info = func_info_realistic[fid]
        print("  -", fid)
        print("    File:", info["file"])
        print("    Signature:", info["signature"])
        if info["docstring"]:
            print("    Docstring:", info["docstring"])
    print()

Bug query: Fix a bug where the loss is not saved correctly.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 3 functions:
  - func:model.forward
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def forward(...):
    Docstring: Run a forward pass through the model.
  - func:data_utils.save_predictions
    File: /tmp/tmpd8_1ngdp/data_utils.py
    Signature: def save_predictions(...):
    Docstring: Save predictions to a JSON file.
  - func:model.train_step
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def train_step(...):
    Docstring: Perform a single training step.

Bug query: Preprocessing does not normalize the data properly.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 3 functions:
  - func:model.forward
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def forward(...):
    Docstring: Run a forward pass through the model.
  - func:train.train_pipeline
    File: /tmp/tmpd8_1ngdp/train.py
    Signature: def train_pipeline(...):
    Docstring: Main training pipeline.
  - func:data_utils.load_data
    File: /tmp/tmpd8_1ngdp/data_utils.py
    Signature: def load_data(...):
    Docstring: Load JSON data from a file.

Bug query: Model forward pass gives wrong predictions.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 3 functions:
  - func:model.forward
    File: /tmp/tmpd8_1ngdp/model.py
    Signature: def forward(...):
    Docstring: Run a forward pass through the model.
  - func:data_utils.save_predictions
    File: /tmp/tmpd8_1ngdp/data_utils.py
    Signature: def save_predictions(...):
    Docstring: Save predictions to a JSON file.
  - func:eval.evaluate
    File: /tmp/tmpd8_1ngdp/eval.py
    Signature: def evaluate(...):
    Docstring: Evaluate the trained model on a dataset.



In [15]:
import pandas as pd

bug_queries = [
    "Fix a bug where the loss is not saved correctly.",
    "Preprocessing does not normalize the data properly.",
    "Model forward pass gives wrong predictions.",
]

rows = []
for bug in bug_queries:
    top_funcs = rank_functions_for_bug(G_realistic, func_info_realistic, bug, top_k=3, alpha=0.7)
    top_short = [fid.split(".", 1)[1] for fid in top_funcs]  # drop 'func:' prefix
    rows.append({
        "Bug query": bug,
        "Top-3 functions": "; ".join(top_short),
    })

df_results = pd.DataFrame(rows)

# Display as a nice table
from IPython.display import display
display(df_results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Bug query,Top-3 functions
0,Fix a bug where the loss is not saved correctly.,forward; save_predictions; train_step
1,Preprocessing does not normalize the data prop...,forward; train_pipeline; load_data
2,Model forward pass gives wrong predictions.,forward; save_predictions; evaluate


In [20]:
import os
import tempfile

def create_larger_demo_repo(base_dir: str):
    # Structure:
    # repo/
    #   config.py
    #   data_utils.py
    #   model.py
    #   train.py
    #   eval.py
    #   loss.py
    #   optim.py
    #   main.py

    with open(os.path.join(base_dir, "config.py"), "w") as f:
        f.write("""
def get_config():
    \"\"\"Return default training configuration.\"\"\"
    return {
        "lr": 1e-3,
        "batch_size": 32,
        "epochs": 10,
        "hidden_dim": 64,
    }
""")

    with open(os.path.join(base_dir, "data_utils.py"), "w") as f:
        f.write("""
import json

def load_data(path: str):
    \"\"\"Load JSON data from a file.\"\"\"
    with open(path, "r") as f:
        return json.load(f)

def preprocess_data(data):
    \"\"\"Preprocess raw data before feeding to model.\"\"\"
    return [x / 255.0 for x in data]

def create_dataloader(data, batch_size: int):
    \"\"\"Create simple batches from data.\"\"\"
    batches = []
    for i in range(0, len(data), batch_size):
        batches.append(data[i:i+batch_size])
    return batches
""")

    with open(os.path.join(base_dir, "model.py"), "w") as f:
        f.write("""
from data_utils import preprocess_data

def build_model(input_dim: int, hidden_dim: int = 64):
    \"\"\"Build a simple MLP model (mock implementation).\"\"\"
    return {"input_dim": input_dim, "hidden_dim": hidden_dim}

def forward(model, x):
    \"\"\"Run a forward pass through the model.\"\"\"
    x = preprocess_data(x)
    h = sum(x) * model["hidden_dim"]
    return h

def compute_logits(model, x):
    \"\"\"Compute logits for classification (mock).\"\"\"
    h = forward(model, x)
    # Mock logits: just scale hidden representation
    return h * 0.1
""")

    with open(os.path.join(base_dir, "loss.py"), "w") as f:
        f.write("""
def mse_loss(pred, target):
    \"\"\"Mean squared error loss.\"\"\"
    return (pred - target) ** 2

def bce_loss(logits, label):
    \"\"\"Binary cross-entropy loss (mock implementation).\"\"\"
    # Simplified: just squared error on logits
    return (logits - label) ** 2
""")

    with open(os.path.join(base_dir, "optim.py"), "w") as f:
        f.write("""
def sgd_step(model, loss, lr=1e-3):
    \"\"\"Mock SGD step updating hidden_dim based on loss.\"\"\"
    model["hidden_dim"] -= lr * loss
    return model
""")

    with open(os.path.join(base_dir, "train.py"), "w") as f:
        f.write("""
from data_utils import load_data, create_dataloader
from model import build_model, forward, compute_logits
from loss import mse_loss, bce_loss
from optim import sgd_step
from config import get_config

def train_pipeline(data_path: str, model_path: str):
    \"\"\"Main training pipeline.\"\"\"
    cfg = get_config()
    data = load_data(data_path)
    loader = create_dataloader(data, cfg["batch_size"])
    model = build_model(input_dim=10, hidden_dim=cfg["hidden_dim"])
    losses = []
    for epoch in range(cfg["epochs"]):
        epoch_loss = 0.0
        for batch in loader:
            # Mock batch: assume batch is list of (x, y)
            for x, y in batch:
                logits = compute_logits(model, x)
                loss = bce_loss(logits, y)
                model = sgd_step(model, loss, lr=cfg["lr"])
                epoch_loss += loss
        losses.append(epoch_loss / max(1, len(loader)))
    # Save losses (mock: just return)
    return model, losses
""")

    with open(os.path.join(base_dir, "eval.py"), "w") as f:
        f.write("""
from data_utils import load_data
from model import build_model, forward, compute_logits
from loss import bce_loss

def evaluate(model_path: str, data_path: str):
    \"\"\"Evaluate the trained model on a dataset.\"\"\"
    model = build_model(input_dim=10, hidden_dim=64)
    data = load_data(data_path)
    preds = []
    total_loss = 0.0
    for x, y in data:
        logits = compute_logits(model, x)
        pred = (logits > 0.5)
        preds.append(pred)
        total_loss += bce_loss(logits, y)
    return preds, total_loss / max(1, len(data))
""")

    with open(os.path.join(base_dir, "main.py"), "w") as f:
        f.write("""
from config import get_config
from train import train_pipeline
from eval import evaluate

def run_experiment(data_path: str, model_path: str):
    \"\"\"Run full training and evaluation experiment.\"\"\"
    cfg = get_config()
    model, losses = train_pipeline(data_path, model_path)
    preds, eval_loss = evaluate(model_path, data_path)
    return {
        "config": cfg,
        "train_losses": losses,
        "eval_loss": eval_loss,
        "predictions": preds,
    }
""")

# Create the larger repo
larger_repo_dir = tempfile.mkdtemp()
create_larger_demo_repo(larger_repo_dir)

print("Larger demo repo created at:", larger_repo_dir)
print("Top-level files:", os.listdir(larger_repo_dir))

Larger demo repo created at: /tmp/tmpam3_ar6n
Top-level files: ['optim.py', 'main.py', 'train.py', 'eval.py', 'config.py', 'data_utils.py', 'model.py', 'loss.py']


In [21]:
# Build graph and extract functions for the larger repo
G_large = build_repo_code_graph(larger_repo_dir)
func_info_large = extract_function_texts(larger_repo_dir)

print("Nodes:", G_large.number_of_nodes())
print("Edges:", G_large.number_of_edges())
print("Functions:", len(func_info_large))

Nodes: 51
Edges: 42
Functions: 13


In [22]:
import pandas as pd
from IPython.display import display

bug_queries_large = [
    "Learning rate is too high and causes unstable training.",
    "Loss computation for classification is incorrect.",
    "Preprocessing does not normalize input data.",
    "Model forward pass gives inconsistent predictions.",
    "Training loop does not save losses properly.",
    "Evaluation returns wrong predictions.",
]

rows = []
for bug in bug_queries_large:
    top_funcs = rank_functions_for_bug(G_large, func_info_large, bug, top_k=3, alpha=0.7)
    top_short = [fid.split(".", 1)[1] for fid in top_funcs]
    rows.append({
        "Bug query": bug,
        "Top-3 functions": "; ".join(top_short),
    })

df_large = pd.DataFrame(rows)
display(df_large)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Bug query,Top-3 functions
0,Learning rate is too high and causes unstable ...,run_experiment; train_pipeline; forward
1,Loss computation for classification is incorrect.,compute_logits; forward; evaluate
2,Preprocessing does not normalize input data.,forward; train_pipeline; create_dataloader
3,Model forward pass gives inconsistent predicti...,forward; evaluate; train_pipeline
4,Training loop does not save losses properly.,forward; train_pipeline; run_experiment
5,Evaluation returns wrong predictions.,evaluate; forward; run_experiment


In [ ]:
## Experiment 2: Larger 8-file repository

We repeat the same procedure on a larger synthetic repository with 8 files and ~20 functions, designed to mimic a small ML project (config, data, model, loss, optimizer, training, evaluation, main script). We define six bug queries covering configuration, loss, preprocessing, forward pass, training, and evaluation.

In [24]:
from PIL import Image, ImageDraw, ImageFont
import os

# Create a blank image
W, H = 1280, 720
img = Image.new("RGB", (W, H), color="#f5f5f5")
draw = ImageDraw.Draw(img)

# Title and subtitle
title = "CodeGraph-Agent"
subtitle = "Graph-based Bug Localization for Gemma 4"

# Use default font (no custom fonts needed)
# We'll just use the default; it’s small but works for a simple card
title_text = title
subtitle_text = subtitle

# Approximate text positioning (no fancy layout)
draw.text((50, 250), title_text, fill="#000000")
draw.text((50, 320), subtitle_text, fill="#444444")

out_path = "codegraph_thumbnail.png"
img.save(out_path)
print("Thumbnail saved as:", out_path)

Thumbnail saved as: codegraph_thumbnail.png
